In [1]:
import os 
import json 
from collections import defaultdict
from typing import List, Dict, Any
import re

papers = [11, 13, 14, 15, 16, 17, 18, 19, 20, 100] # read the annotated overlapping paper IDs 
overlap_path = "../../annotated_data/citation_annotations/annotations/overlap"
ann_path = "../../annotated_data/citation_annotations/annotations"
text_folder = "../../to_annotate"
total_data = []

for files in os.listdir(overlap_path):
    if files.endswith(".json"):
        file_path = os.path.join(overlap_path, files)
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for item in data:
            filename = item["file"]
            with open(os.path.join(text_folder, filename), "r", encoding="utf-8") as f:
                full_text = f.read()
            total_data.append({"file": item["file"], "start": item["start"], "end": item["end"], "label": item["label"], 'text': item['text'], 'full_text': full_text.split("References")[0]})
    

for ann in os.listdir(ann_path):
    print("Processing folder:", ann)
    for file in os.listdir(os.path.join(ann_path, ann)):
        if file.endswith(".json") and int(file.split("_")[1].split(".")[0]) not in papers:    
            file_path = os.path.join(ann_path, ann, file)
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            for item in data:
                filename = item["file"]
            with open(os.path.join(text_folder, filename), "r", encoding="utf-8") as f:
                full_text = f.read()
            total_data.append({"file": item["file"], "start": item["start"], "end": item["end"], "label": item["label"], 'text': item['text'], 'full_text': full_text.split("References")[0]})
        
with open("../../annotated_data/goldset_sorted.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for item in data:
    for label in item:
        filename = label["filename"]
        with open(os.path.join(text_folder, filename), "r", encoding="utf-8") as f:
            full_text = f.read()
        total_data.append({"file": label["filename"], "start": label["start"], "end": label["end"], "label": label["label"], 'text': label['text'], 'full_text': full_text.split("References")[0]})


_PAPER_NUM_RE = re.compile(r"paper_(\d+)\.txt")

def _paper_key(filename: str) -> int:
    """
    Extract numeric index from 'paper_n.txt' for sorting.
    Falls back to +inf if pattern doesn't match.
    """
    m = _PAPER_NUM_RE.match(filename)
    return int(m.group(1)) if m else float("inf")

def group_spans_by_file(records: List[Dict[str, Any]]):
    grouped = defaultdict(list)

    for r in records:
        grouped[r["file"]].append({
            "start": int(r["start"]),
            "end": int(r["end"]),
            "label": r["label"],
            "text": r['text'],
            "full_text": r['full_text']
        })

    # Sort by numeric paper index
    sorted_grouped = dict(
        sorted(grouped.items(), key=lambda item: _paper_key(item[0]))
    )

    return sorted_grouped
total_data = group_spans_by_file(total_data)

Processing folder: Iman
Processing folder: Ekaterina
Processing folder: overlap
Processing folder: Iraa
Processing folder: Ed


In [2]:
#train and eval split
import random
import re
from typing import Dict, List, Any, Tuple
import json 

PaperData = Dict[str, List[Dict[str, Any]]]

_PAPER_RE = re.compile(r"paper_(\d+)\.txt$")

def paper_num(name: str) -> int:
    m = _PAPER_RE.search(name)
    return int(m.group(1)) if m else 10**18  # unknowns go to end

def split_by_paper(
    data_by_paper: PaperData,
    train_ratio: float = 0.5,
    dev_ratio: float = 0.5,
    eval_ratio: float = 0,
    seed: int = 42,
) -> Tuple[PaperData, PaperData, PaperData]:
    assert abs((train_ratio + dev_ratio + eval_ratio) - 1.0) < 1e-9

    keys = sorted(data_by_paper.keys(), key=paper_num)

    rng = random.Random(seed)
    rng.shuffle(keys)

    n = len(keys)
    n_train = int(n * train_ratio)
    n_dev = int(n * dev_ratio)
    # eval gets the remainder to avoid rounding issues
    train_keys = keys[:n_train]
    dev_keys = keys[n_train:n_train + n_dev]
    eval_keys = keys[n_train + n_dev:]

    train = {k: data_by_paper[k] for k in train_keys}
    dev = {k: data_by_paper[k] for k in dev_keys}
    eval_ = {k: data_by_paper[k] for k in eval_keys}

    return train, dev, eval_

eval_data, for_synth_data, _ = split_by_paper(total_data)

def save_split(split: dict, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(split, f, ensure_ascii=False, indent=2)

save_split(eval_data, "for_eval.json")
save_split(for_synth_data, "for_synth_sampling.json")


In [1]:
import json

with open("samples/for_synth_sampling.json", 'r') as f:
    for_synth_data = json.load(f)

In [2]:
coherence_samples = []
unsupp_claim = []
lack_synth = []
format = []

for ann in for_synth_data:
    for item in for_synth_data[ann]: 
        if item['label'].lower() == 'coherence':
            coherence_samples.append(item)
        elif item['label'].lower() == 'unsupported claim':
            unsupp_claim.append(item)
        elif item['label'].lower() == 'lacks synthesis':
            lack_synth.append(item)
        elif item['label'].lower() == 'format':
            format.append(item)

print("Annotated data stats:")
print(f"{len(coherence_samples)} Coherence samples")
print(f"{len(unsupp_claim)} Unsupported claim samples")
print(f"{len(lack_synth)} Lacks Synthesis samples")
print(f"{len(format)} Format samples")

Annotated data stats:
14 Coherence samples
60 Unsupported claim samples
10 Lacks Synthesis samples
18 Format samples


In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os 

key_path = 'keys.env'
load_dotenv(dotenv_path=key_path) 
api_key = os.getenv("API_KEY_2")
client = OpenAI(api_key=api_key)

def ask_gpt(prompt):
    completion = client.chat.completions.create(
        model="gpt-5",
        messages=[
            {"role": "assistant", "content": "You are an expert in academic writing and citation analysis."},
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return completion.choices[0].message.content

In [4]:
format_def = """Issues in citation formatting such as a missing bracket and using the wrong style of citing.
    a) Due to preprocessing errors of the source dataset, some words contain hyphens that do not require it, and some are missing hyphens where it is required. Please ignore these types of formatting issues.
    b) Highlight the word/citation in which the formatting issue occurs in and not only the issue within the word/citation.
    c) Formatting issues appear as either citations or parts of a citation.
    Examples of formatting issues include:
        i) Narrative citation missing year: “Vatswani et al.” -> should be “Vatswani et al. (2020)”
        ii) Wrong citation style: “In (Vatswani et al., 2019)” -> should be “in Vatswani et al. (2019)”
        iii) Wrong use of footnotes: "Vastwani et al. 1" -> should include the year or be reformatted as a proper footnote."""

unsupp_def = """claim about prior work or statistics w/o citation or evidence. 
    a) The author should cite at every first mention of a study, paper, shared task, competition or dataset.
    b) Specific information to a niche topic, despite sounding like it should be known in that topic of study, should be cited.
    c) If a claim is made and is obvious to be a natural deduction from previous statements through common sense (i.e not requiring expert knowledge), then this claim does not fall under ‘Unsupported claim’. For example:
        i) “However, creating a large and suitable set of questions for supporting narrative comprehension is both time-consuming and cognitively demanding.” -> it is obvious that creating a dataset is time consuming and mentally demanding.
    d) Any mention of “recent works” should be backed up with citations to the works.
    e) Unsupported claim issues appear as segments, phrases, sub-sentences or full sentences.
    Examples of unsupported claims include:
        i) Missing citations for mentions of 'recent works': “and there are many recent works that explore this topic”,
        ii) Mention of a previous work and claim without citation: “..., while in a previous study, the authors claim …”,
        iii) Mentioning of a specific setup of a task without citation to the work: ".. BERT was used in an AES task trained on essays .." """

lacksynth_def = """occurs when either:
    a) The author describes or cites papers without connecting them to their own work/argument 
    b) Or only follows up the summary of previous works with their own contribution without explicitly highlighting the gap their work intends to research.
    c) It does not articulate the author's perspective or motivation.
    d) A lack of argument/opinion in the first paragraph is permissible as it serves to be the foundation of the author's argument 
    e) Lacks synthesis issues appear either as single sentences or multiple sentences.
    Examples of lack of synthesis include:
        i) No elaboration of own contribution/argument:"Following early neural approaches to question answering, many subsequent studies adopt a pipeline architecture consisting of retrieval and comprehension components. The retrieval component focuses on identifying relevant documents or passages from a large corpus, while the comprehension component extracts an answer span from the retrieved text. Initial models relied on recurrent neural networks with attention mechanisms to encode questions and contexts (Seo et al., 2017; Wang et al., 2017)."
        ii)  No explanation of the cited works and relation to their own work: “Recently, several studies have explored the use of prompting techniques with pre-trained language models to influence model outputs or access latent knowledge (Brown et al., 2020; Gao et al., 2021; Liu et al., 2021; Wei et al., 2022).” """

coherence_def = """connection between cited works is abrupt, lacking relation to each other. It is unclear how one mentioned work is relevant to a prior mentioned work. 
    a) Sentences are not transitioned from one to another.
    b) The relationship between sentences describing papers is implied but not explicitly stated.
    c) Coherence issues appear only as multiple sentences.
    Examples of coherence issues include:
        i) Relation between mentioned works is not explicit: “Smith (2020) identified a relationship between personal belief systems and ethical decision-making frameworks. Moral foundation theory proposes several core dimensions of moral reasoning, including harm, fairness, and authority (Jones, 2015). Audience adaptation has been explored in computational argumentation. Lee et al. (2019) applied moral categories to argument generation tasks. Human annotators often disagree when labeling moral dimensions in text (Nguyen et al., 2018).”
        ii) Lack of transitions between sentences: “Recent studies have explored various techniques for enhancing model performance. Smith et al. (2020) introduced a novel architecture that significantly improves accuracy on benchmark datasets. Additionally, Johnson and Lee (2019) proposed a data augmentation method that increases training data diversity.” 
        iii) No explanation of the cited works and relation to their own work: “Recently, several studies have explored the use of prompting techniques with pre-trained language models to influence model outputs or access latent knowledge (Brown et al., 2020; Gao et al., 2021; Liu et al., 2021; Wei et al., 2022).” """

Should the synthetic samples reflect the same ratio of categories found in samples?

In [5]:
import json 
import numpy as np

with open("stats.json", 'r') as f:
    stats= json.load(f)

class CategoryDistribution:
    def __init__(self, stats):
        self.lengths = np.array(stats["lengths"])
        self.spans_per_doc = np.array(stats["spans_per_document"])
        self.sentences_per_doc = np.array(stats["sentences_per_document"])

    def sample_num_spans(self):
        return int(np.random.choice(self.spans_per_doc))

    def sample_span_lengths(self, k):
        return list(np.random.choice(self.lengths, size=k, replace=True))

    def sample_num_sentences(self):
        return int(np.random.choice(self.sentences_per_doc))

In [6]:
from tqdm import tqdm
import json
import re
import random

total = {"Unsupported claim": [], "Format": [], "Coherence": [], "Lacks synthesis": []}
categories = ["Coherence"]

total_amount = {
    "Unsupported claim": 300,
    "Format": 300,
    "Coherence": 300,
    "Lacks synthesis": 275,
}

BATCH_SIZE = 5
random.seed(42)

CATEGORY_CONSTRAINTS = {
    "Coherence": """- The document should mention at least 3 distinct prior works in sequence.
- The new span should create an implicit topic shift or weak connection WITHOUT explicit transition markers.
- Avoid discourse connectors such as "however", "in contrast", "similarly", "therefore", "moreover" inside the new span.""",
    "Lacks synthesis": """- The new span should read like a literature list: mention at least 4 works/datasets/methods.
- The span must NOT include author stance or synthesis (avoid: "we argue", "we propose", "this suggests", "overall").
- The paragraph should feel complete but still lacks the author's own connecting argument.""",
    "Unsupported claim": """- The new span must include a generalizing claim about prior work without citation/evidence.
- The claim should reference a specific method/dataset/trend (e.g., "two-stage retriever-reader") but NOT cite who did it.
- Use realistic academic phrasing such as "recent studies", "prior work", "widely adopted approaches".""",
    "Format": """- The new span must be ONLY the citation text (not the surrounding sentence).
- Introduce realistic formatting mistakes: missing bracket, wrong style (narrative vs parenthetical), truncated year, etc.
- Keep it plausible under the paper's citation style.""",
}


def trim_extremes(values, lower=5, upper=95):
    lo = np.percentile(values, lower)
    hi = np.percentile(values, upper)
    return [v for v in values if lo <= v <= hi]

def parse_gpt_json(response):
    json_structure = json.loads(response)
    return json_structure

def pick_def_and_samples(category):
    if category == "Unsupported claim":
        return unsupp_def, unsupp_claim
    elif category == "Format":
        return format_def, format
    elif category == "Coherence":
        return coherence_def, coherence_samples
    elif category == "Lacks synthesis":
        return lacksynth_def, lack_synth
    else:
        raise ValueError(f"Unknown category: {category}")

def approx_sentence_count(text):
    parts = re.split(r"[.!?]\s+", text.strip())
    parts = [p for p in parts if p.strip()]
    return len(parts)

def validate_item(item, category):
    spans = item["spans"]
    doc = item["document"]
    reason = item["reason"]
    # document shouldn't be extremely short
    if approx_sentence_count(doc) < min(stats[category]['sentences_per_document']):
        return None

    return {"spans": spans, "document": doc, "reason": reason}


for category in categories:
    definition, samples = pick_def_and_samples(category)
    constraints = CATEGORY_CONSTRAINTS[category]

    span_len = trim_extremes(stats[category]['lengths'])
    num_spans = trim_extremes(stats[category]['spans_per_document'])
    num_sents = trim_extremes(stats[category]['sentences_per_document'])
    stats[category]['lengths'] = span_len
    stats[category]['spans_per_document'] = num_spans
    stats[category]['sentences_per_document'] = num_sents
    dist = CategoryDistribution(stats[category])

    print(f"\nGenerating synthetic samples for category: {category}")

    target = int(total_amount[category])
    run = 0
    pbar = tqdm(total=target, initial=0, desc=f"{category}")

    example_docs = []
    for s in samples:
        ft = s.get("full_text", "")
        if isinstance(ft, str) and ft.strip():
            example_docs.append(ft)

    while len(total[category]) < target:
        run += 1
        remaining = target - len(total[category])
        this_batch = min(BATCH_SIZE, remaining)

        num_pairs = max(1, (len(samples) - 1) // 2)
        pair_idx = (run - 1) % num_pairs
        i = pair_idx * 2

        num_spans = dist.sample_num_spans()
        span_lengths = dist.sample_span_lengths(num_spans)
        num_sentences = dist.sample_num_sentences()

        prompt = f"""
You are an expert in academic writing and citation analysis.

Your task is to generate EXACTLY {this_batch} realistic academic documents.
Each document must contain EXACTLY {num_spans} spans exhibiting the issue "{category}".

Definition of "{category}":
{definition}

Span constraints:
- Each document must contain EXACTLY {num_spans} problematic spans.
- The spans must have word lengths: {span_lengths}.
  (The first span should have approximately the first length, the second span the second length, etc.)
- The spans must be distributed throughout the document.
  DO NOT concentrate them at only the beginning or the end.

Document constraints:
- The document should resemble an Introduction or Related Work section.
- Maintain realistic academic tone and structure.
- The document should contain {num_sentences} sentences.
- You can have templated phrasing.
- Do not copy the example documents verbatim.

Category-specific constraints:
{constraints}

You are given real examples below to guide style and structure:

Span: "{samples[i]['text']}"
Document:
{samples[i]['full_text']}

Span: "{samples[i+1]['text']}"
Document:
{samples[i+1]['full_text']}"

End of examples.

Now generate EXACTLY {this_batch} new documents.
Each document must contain EXACTLY {num_spans} spans satisfying the above constraints.

Return a JSON array with EXACTLY {this_batch} items.

Each item must have the format:
{{
  "spans": [span1, span2, ...],
  "document": full document text,
  "reason": Brief explanation of why each span satisfies the definition.
}}

Important:
- The spans list must contain EXACTLY {num_spans} spans.
- Do not include any additional commentary.
- Output only valid JSON.
""".strip()
        response = ask_gpt(prompt)
        gpt_response = parse_gpt_json(response)

        if not gpt_response:
            print(f"[WARN] Got 0 JSON items for {category} on run {run}. Retrying...")
            continue

        cleaned = []
        for item in gpt_response:
            valid = validate_item(item, category)
            if valid is not None:
                cleaned.append(valid)
            else:
                continue
            
        remaining = target - len(total[category])
        cleaned = cleaned[:remaining]

        total[category].extend(cleaned)
        pbar.update(len(cleaned))

        if run % 5 == 0:
            print(f"{category}: {len(total[category])}/{target} after run {run}")
            with open(f"{category}_synth_samples.json", "w", encoding="utf-8") as f:
                json.dump(total[category], f, indent=2, ensure_ascii=False)

    pbar.close()
    print(f"Generated {len(total[category])} samples for category {category}")

with open("synthetic_samples_4.json", "w", encoding="utf-8") as f:
    json.dump(total, f, indent=2, ensure_ascii=False)

print("Done. Wrote per-category files + synthetic_samples_4.json")


Generating synthetic samples for category: Coherence


Coherence:   8%|▊         | 25/300 [19:02<3:27:54, 45.36s/it]

Coherence: 25/300 after run 5


Coherence:  17%|█▋        | 50/300 [37:53<3:16:40, 47.20s/it]

Coherence: 50/300 after run 10


...

Coherence:  67%|██████▋   | 200/300 [2:10:24<53:46, 32.26s/it]  

Coherence: 200/300 after run 40


Coherence:  75%|███████▌  | 225/300 [2:25:08<45:52, 36.71s/it]

Coherence: 225/300 after run 45


Coherence:  77%|███████▋  | 230/300 [2:28:40<44:49, 38.43s/it]

In [1]:
import json 

with open("Unsupported claim_synth_samples.json", 'r') as f:
    unsupp_data = json.load(f)

with open("Format_synth_samples.json", 'r') as f:
    format_data = json.load(f)

with open("Lacks synthesis_synth_samples.json", 'r') as f:
    lacksynth_data = json.load(f)

with open("Coherence_synth_samples.json", 'r') as f:
    coherence_data = json.load(f)

total = {'Unsupported claim': unsupp_data, 'Format': format_data, 'Coherence': coherence_data, 'Lacks synthesis': []}

In [2]:
with open("samples/synthetic_samples_5.json", "w", encoding="utf-8") as f:
    json.dump(total, f, ensure_ascii=False, indent=2)